# AI Lara — Qwen LoRA/QLoRA Starter
Notebook awal untuk menyiapkan lingkungan dan menguji Qwen2.5-1.5B-Instruct di Google Colab. Jalankan sel dari atas ke bawah. Pelatihan adapter memerlukan GPU. Notebook ini belum menghasilkan APK Android.

## 1. Periksa runtime dan GPU

In [ ]:
import sys, torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA tersedia:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Instal pustaka
Jalankan sekali. Jika Colab meminta restart runtime, restart lalu lanjutkan dari sel pertama.

In [ ]:
%pip -q install -U "transformers>=4.46,<5" "datasets>=3.0" "peft>=0.13" "accelerate>=1.1" "bitsandbytes>=0.44" "safetensors"
print('Instalasi selesai')

## 3. Muat model Qwen dalam 4-bit

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map='auto',
)
print('Model Qwen berhasil dimuat')

## 4. Uji respons model

In [ ]:
messages = [
    {'role': 'system', 'content': 'Kamu adalah AI Lara, asisten yang membantu merancang aplikasi Android.'},
    {'role': 'user', 'content': 'Jelaskan langkah awal membuat aplikasi catatan sederhana untuk Android.'},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=250, do_sample=True, temperature=0.7, top_p=0.9)
print(tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))

## 5. Siapkan dataset contoh untuk AI Lara
Sel ini hanya menyiapkan lima contoh instruksi-respons. Ini belum melatih model; kita akan menambahkan konfigurasi LoRA setelah dataset ini berhasil dibuat.

In [ ]:
from datasets import Dataset

data = [
    {
        'instruction': 'Buat aplikasi catatan sederhana untuk Android.',
        'response': 'Buat proyek Android dengan Kotlin. Tampilkan daftar catatan, kolom input, tombol simpan, dan penyimpanan lokal.'
    },
    {
        'instruction': 'Buat aplikasi daftar tugas Android.',
        'response': 'Buat aplikasi Android dengan Kotlin yang memiliki daftar tugas, tombol tambah, fitur tandai selesai, dan penyimpanan lokal.'
    },
    {
        'instruction': 'Bagaimana membuat APK Android?',
        'response': 'Siapkan proyek Android yang lengkap, gunakan Gradle, lalu jalankan tugas assembleDebug untuk membangun APK debug.'
    },
    {
        'instruction': 'Buat aplikasi kalkulator Android.',
        'response': 'Buat antarmuka kalkulator dengan tombol angka, operasi matematika, dan area hasil.'
    },
    {
        'instruction': 'Apa yang dibutuhkan AI Lara untuk menghasilkan APK?',
        'response': 'AI Lara perlu menghasilkan kode sumber proyek Android, menyiapkan konfigurasi build, menjalankan Gradle, lalu memeriksa berkas APK yang dihasilkan.'
    }
]

dataset = Dataset.from_list(data)
print('Jumlah contoh:', len(dataset))
print(dataset[0])

## Tahap berikutnya
Setelah model dan dataset contoh berhasil diuji, tahap berikutnya adalah menyiapkan format prompt dan konfigurasi pelatihan LoRA. Dataset lima contoh ini hanya untuk memastikan alur kerja berjalan; pelatihan yang bermanfaat membutuhkan lebih banyak contoh berkualitas. **Adapter LoRA bukan APK**; pembuatan APK Android adalah tahap proyek terpisah.

## 6. Pasang LoRA pada model
Sel ini menyiapkan adapter LoRA untuk Qwen yang sudah dimuat di atas. Jalankan setelah sel dataset. Ini belum memulai pelatihan.

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 7. Format data untuk pelatihan
Lima contoh di atas hanya untuk smoke test agar pipeline bisa diuji. Model tidak akan menjadi pembuat APK yang andal dengan dataset sekecil ini.

In [ ]:
def format_example(example):
    messages = [
        {"role": "system", "content": "Kamu adalah AI Lara, asisten yang mengubah permintaan aplikasi Android menjadi rancangan teknis yang jelas dan aman."},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

formatted = dataset.map(format_example)
tokenized_dataset = formatted.map(
    lambda batch: tokenizer(batch["text"], truncation=True, max_length=512),
    batched=True,
    remove_columns=formatted.column_names,
)
print("Contoh siap dilatih:", len(tokenized_dataset))
print(tokenized_dataset[0].keys())

## 8. Jalankan pelatihan percobaan LoRA
Gunakan GPU T4. Pelatihan ini hanya memverifikasi alur teknis; hasilnya bukan model produksi. Jika kehabisan memori, mulai ulang runtime dan pastikan tidak memuat model dua kali.

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="/content/ai-lara-lora-output",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to="none",
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)
trainer.train()

## 9. Simpan adapter LoRA ke Google Drive
Sel ini menyimpan adapter ke Drive supaya tidak hilang ketika sesi Colab berakhir. Jalankan setelah `trainer.train()` selesai. Saat diminta, pilih akun Google dan izinkan akses Drive.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
ADAPTER_DIR = "/content/drive/MyDrive/AI-Lara/ai-lara-lora-adapter"
Path(ADAPTER_DIR).mkdir(parents=True, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Adapter LoRA berhasil disimpan!")
print("Lokasi:", ADAPTER_DIR)
print("Catatan: adapter ini bukan APK Android.")

## Tahap setelah notebook
- Evaluasi adapter dengan banyak contoh yang belum pernah dilihat saat pelatihan.
- Buat dataset yang lebih besar, beragam, dan diperiksa manusia.
- Hubungkan model ke backend hanya setelah ada evaluasi dan batas keamanan.
- Generator Android dan workflow APK ada di bagian proyek terpisah.
- Jangan menganggap teks atau kode yang dihasilkan aman tanpa pengujian.

## 10. Uji AI Lara setelah pelatihan
Jalankan setelah sel penyimpanan. Ini menguji apakah model yang sedang berada di memori dapat menjawab permintaan rancangan aplikasi. Karena dataset masih lima contoh, hasilnya belum membuktikan kualitas model produksi.

In [ ]:
messages = [
    {"role": "system", "content": "Kamu adalah AI Lara, asisten pembuat rancangan aplikasi Android. Jawab dalam bahasa Indonesia dan jangan mengaku sudah membangun APK jika belum."},
    {"role": "user", "content": "Rancang aplikasi catatan Android dengan fitur tambah catatan, daftar catatan, dan penyimpanan lokal. Jelaskan komponen yang diperlukan."}
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
model.eval()
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=250, do_sample=False)
answer = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("HASIL UJI AI LARA:")
print(answer)